# Archived custom temporal split

PLINDER publishes annotations and representative-cover assignments rather than one mandatory train/test split. This example builds a small, reproducible split table with a release-date cutoff and a ligand-family separation rule.

In [ ]:
import os

os.environ.setdefault("PLINDER_RELEASE", "2026-07")
os.environ.setdefault("PLINDER_RELEASE_NUMBER", "1")

In [ ]:
import pandas as pd

from plinder.core import query_table

## Select the candidate systems

The annotation table has one row per ligand. Restricting this example to systems with one proper ligand gives one row per system. Entry release dates are stored once in `entry_metadata`; requesting the date selects that table automatically. The pocket-residue and interaction bounds are example statistical criteria; choose bounds suited to the intended evaluation.

In [ ]:
cluster_col = (
    "tanimoto_similarity_ecfp4_1024__70__ligand__set_cover"
)
columns = [
    "system_id",
    "entry_pdb_id",
    "ligand_id",
    "ligand_unique_ccd_code",
    "system_pass_validation_criteria",
    "system_proper_num_pocket_residues",
    "system_proper_num_interactions",
    "entry_release_date",
    cluster_col,
]
systems = query_table(
    "annotation",
    columns=columns,
    filters=[
        ("ligand_is_proper", "==", True),
        ("system_proper_num_ligand_chains", "==", 1),
        ("system_proper_num_pocket_residues", ">=", 5),
        ("system_proper_num_pocket_residues", "<=", 100),
        ("system_proper_num_interactions", ">=", 3),
    ],
)
systems["entry_release_date"] = pd.to_datetime(
    systems["entry_release_date"]
)
assert not systems["system_id"].duplicated().any()
systems.shape

## Keep only ligand families that are entirely post-cutoff

A system is eligible for the test set only when it was released after the cutoff and no pre-cutoff candidate has the same 70% Tanimoto set-cover label. Post-cutoff systems in mixed families, and systems without a cover label, are marked `removed`.

In [ ]:
cutoff = pd.Timestamp("2021-09-30")
is_post_cutoff = systems["entry_release_date"] > cutoff
is_train_period = systems["entry_release_date"].le(cutoff)
cluster_has_pre_cutoff = (
    is_train_period
    .groupby(systems[cluster_col], dropna=False)
    .transform("any")
)
is_pure_post_cutoff = (
    is_post_cutoff
    & systems[cluster_col].notna()
    & ~cluster_has_pre_cutoff
)

In [ ]:
systems["split"] = "removed"
systems.loc[is_train_period, "split"] = "train"
systems.loc[is_pure_post_cutoff, "split"] = "test"
systems["split"].value_counts()

Check the two conditions directly: test systems are post-cutoff, and their published ligand-family labels do not occur in training.

In [ ]:
test_rows = systems[systems["split"] == "test"]
train_clusters = set(
    systems.loc[systems["split"] == "train", cluster_col].dropna()
)
test_clusters = set(test_rows[cluster_col].dropna())
assert (test_rows["entry_release_date"] > cutoff).all()
assert systems.loc[systems["split"] == "train", "entry_release_date"].le(cutoff).all()
assert train_clusters.isdisjoint(test_clusters)

Create the compact table expected by downstream code. Additional columns can be retained for auditing.

In [ ]:
split_table = systems[["system_id", "split"]].copy()
split_table.head()

A set-cover label groups each member with a chosen representative; it is not a transitive similarity component. This recipe prevents shared published labels across train and test, but does not prove that every cross-split ligand pair is below 70% Tanimoto. Use the all-vs-all ligand scores when that stricter pairwise guarantee is required. The same pattern can combine ligand, pocket, and interface cover labels depending on the leakage definition.